In [ ]:
%load_ext autoreload
%autoreload 2

# Custom allocator

Subclass `BaseAllocator`, implement the `_allocate` hook, and the registry
picks the allocator up automatically — usable by name everywhere, including
`om.allocate` and the benchmark harness.

In [ ]:
import omnimalloc as om
from omnimalloc.allocators import BaseAllocator, available_allocators
from omnimalloc.benchmark.sources import RandomSource

`_allocate` receives validated, non-empty allocations with unique ids and
returns them with offsets assigned. The one constraint to satisfy:
*conflicting* allocations — those whose lifetimes can coexist, as reported
by `om.conflicts` — must occupy disjoint address ranges.

This allocator places largest-first, each allocation at the lowest offset
not blocked by an already-placed conflict. Because it consumes only the
conflict relation, it also accepts vector-clock lifetimes
(`supports_vector_time = True`).

In [ ]:
class LargestFirstAllocator(BaseAllocator):
    """Place allocations largest-first, each at the lowest conflict-free offset."""

    supports_vector_time = True

    def _allocate(
        self, allocations: tuple[om.Allocation, ...]
    ) -> tuple[om.Allocation, ...]:
        conflict_map = om.conflicts(allocations)
        sizes = {alloc.id: alloc.size for alloc in allocations}
        offsets: dict[om.IdType, int] = {}
        for alloc in sorted(allocations, key=lambda a: a.size, reverse=True):
            blockers = sorted(
                (offsets[other], offsets[other] + sizes[other])
                for other in conflict_map[alloc.id]
                if other in offsets
            )
            offset = 0
            for lower, upper in blockers:
                if lower - offset >= alloc.size:
                    break
                offset = max(offset, upper)
            offsets[alloc.id] = offset
        return tuple(alloc.with_offset(offsets[alloc.id]) for alloc in allocations)

Defining the class registered it: the `Allocator` role token is stripped and
the rest snake_cased, so it answers to `"largest_first"`.

In [ ]:
assert "largest_first" in available_allocators()

pool = RandomSource(num_allocations=200).get_pool()
pool = om.allocate(pool, "largest_first", validate=True)
print(
    f"peak memory {pool.size}, lower bound {pool.pressure}, "
    f"efficiency {pool.efficiency:.3f}"
)

Compare it against a few built-in allocators, then plot its placement.

In [ ]:
for name in ("naive", "greedy_by_size", "omni", "largest_first"):
    print(f"{name:>14}: peak {om.allocate(pool, name).size}")

om.plot_allocation(pool)